<a href="https://colab.research.google.com/github/zach8421/portable_waste_sorting/blob/main/dc/dc_waste_sorting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portable Waste-Sorting Information for Washington, DC

**IMT 542 - DC implementation**

This notebook adapts the Seattle portable waste-sorting workflow to Washington, DC. It pulls material guidance from the ReCollect API behind Zero Waste DC's **What Goes Where** tool, normalizes each record into a reusable item-level structure, visualizes the inferred disposal streams, and exposes a simple lookup function.

The DC source does not publish the same taxonomy and per-stream fields as Seattle Public Utilities. Instead, DC's records contain page variables and reusable option templates such as `wizard_recycling`, `wizard_trash`, `wizard_bulk_trash_collection`, and `wizard_drop-off_*`. This notebook maps those templates into the same stream vocabulary used by the Seattle notebook: `recycling`, `compost`, `garbage`, `hazardous`, `dropoff`, `special`, and `donate`.

In [1]:

# Install notebook dependencies. Colab usually has these pre-installed, but this keeps the notebook portable.
%pip install -q pandas plotly requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:

import json
import re
from collections import defaultdict
from datetime import datetime, timezone
from html import unescape
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import requests

## 1. Fetch the DC material records

The ReCollect endpoint returns material pages for Washington, DC. We request the records in pages of 100 with an `offset` parameter, which is more reliable than assuming the API will honor one very large `limit` value.

If the live request fails and a raw snapshot already exists in `data/dc_recollect_raw_pages.json`, the notebook falls back to that local snapshot so the rest of the workflow can still run offline.

In [3]:

SOURCE_URL = "https://api.recollect.net/api/areas/WashingtonDC/services/220/pages"
PAGE_SIZE = 100
if Path("dc_waste_sorting.ipynb").exists() or Path("dc_recollect_pull.py").exists():
    DATA_DIR = Path("data")
elif Path("dc/dc_waste_sorting.ipynb").exists():
    DATA_DIR = Path("dc") / "data"
else:
    DATA_DIR = Path("data")
RAW_SNAPSHOT = DATA_DIR / "dc_recollect_raw_pages.json"
PORTABLE_SNAPSHOT = DATA_DIR / "dc_items_portable.json"

BASE_PARAMS = {
    "suggest": "",
    "type": "material",
    "set": "default",
    "include_links": "true",
    "locale": "en-US",
    "accept_list": "true",
    "limit": PAGE_SIZE,
}


def fetch_all_pages():
    all_items = []
    offset = 0
    last_url = None

    while True:
        params = {**BASE_PARAMS, "offset": offset}
        response = requests.get(SOURCE_URL, params=params, timeout=30)
        response.raise_for_status()
        batch = response.json()
        last_url = response.url

        if not batch:
            break

        all_items.extend(batch)
        print(f"offset={offset}: +{len(batch)} records (total={len(all_items)})")

        if len(batch) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

    return all_items, last_url


def load_raw_records():
    DATA_DIR.mkdir(exist_ok=True)
    try:
        raw_items, last_url = fetch_all_pages()
        with RAW_SNAPSHOT.open("w", encoding="utf-8") as f:
            json.dump(raw_items, f, indent=2, ensure_ascii=False)
        source_mode = "live API"
    except requests.RequestException as exc:
        if not RAW_SNAPSHOT.exists():
            raise
        print(f"Live fetch failed ({exc}). Loading cached raw snapshot from {RAW_SNAPSHOT}.")
        with RAW_SNAPSHOT.open(encoding="utf-8") as f:
            raw_items = json.load(f)
        last_url = str(RAW_SNAPSHOT)
        source_mode = "cached raw snapshot"

    return raw_items, last_url, source_mode


raw_items, source_url, source_mode = load_raw_records()
print(f"Loaded {len(raw_items)} raw records from {source_mode}")

offset=0: +100 records (total=100)


offset=100: +100 records (total=200)


offset=200: +100 records (total=300)


offset=300: +100 records (total=400)


offset=400: +68 records (total=468)
Loaded 468 raw records from live API


## 2. Transform records into the portable schema

Seattle's API has explicit fields like `Garbage`, `Recycling`, and `Hazardous`. DC's ReCollect records instead attach reusable instruction blocks. The `infer_streams_for_option()` function maps those blocks to the shared stream vocabulary.

A generic "comprehensive list of what can be recycled" link appears on many DC records, so the stream inference ignores that boilerplate instead of treating every item as recyclable.

In [4]:

STREAM_PRIORITY = ["hazardous", "recycling", "compost", "garbage", "dropoff", "special", "donate"]
STREAM_DISPLAY = {
    "recycling": "Recycling",
    "compost": "Compost / organics",
    "garbage": "Trash",
    "hazardous": "Household hazardous waste",
    "dropoff": "Drop-off",
    "special": "Special collection",
    "donate": "Donate / repair / reuse",
    "other": "Uncategorized",
}

STREAM_COLOUR = {
    "recycling": "#2b8a3e",
    "compost": "#8a6d1a",
    "garbage": "#495057",
    "hazardous": "#c92a2a",
    "dropoff": "#1864ab",
    "special": "#6741d9",
    "donate": "#0c8599",
    "other": "#868e96",
}

TEMPLATE_STREAM_RULES = {
    "hazardous": [
        "household_hazardous_waste", "battery", "batteries", "paintcare",
        "medications", "sharps",
    ],
    "donate": [
        "donate", "repair", "reuse", "fix-it", "edible_food", "cars",
        "clothes", "toys", "household_items",
    ],
    "compost": [
        "food_waste", "yard_waste", "leaf_collection", "holiday_tree",
        "grasscycling",
    ],
    "special": [
        "bulk_trash", "holiday_tree_collection", "leaf_collection", "yard_waste",
    ],
    "recycling": [
        "wizard_recycling", "ecycle_dc", "e-cycle", "metals_recycling", "brita",
    ],
    "garbage": ["wizard_trash"],
    "dropoff": [
        "drop-off", "drop_off", "dropoff", "fort_totten", "call2recycle",
        "best_buy", "home_depot", "mom_s", "paintcare", "take-back", "depot",
    ],
}

TEXT_STREAM_RULES = {
    "hazardous": [
        "household hazardous waste", "hazardous waste", "safe handling",
    ],
    "dropoff": [
        "drop off this item", "drop-off location", "drop-off locations",
        "drop off batteries", "drop off", "no curbside collection",
    ],
    "special": [
        "bulk trash collection", "make an appointment", "call 311",
    ],
    "donate": ["donate", "repair this item", "reuse"],
    "compost": ["food waste drop-off", "yard waste", "compost"],
}

BOILERPLATE_RECYCLING_TEXT = "comprehensive list of what can be recycled in the district"


def strip_html(value):
    """Convert ReCollect HTML snippets to readable plain text."""
    if not value:
        return ""
    value = re.sub(r"<li[^>]*>", " * ", value, flags=re.I)
    value = re.sub(r"<br\s*/?>", " ", value, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", value)
    text = unescape(text)
    return re.sub(r"\s+", " ", text).strip()


def get_variable(item, name, locale="en-US"):
    """Read a named ReCollect variable, handling localized dict values."""
    variables = item.get("opts", {}).get("variables", [])
    for var in variables:
        if var.get("name") == name:
            value = var.get("value", "")
            if isinstance(value, dict):
                return value.get(locale) or value.get("en") or ""
            return value or ""
    return ""


def extract_options(item):
    """Collect reusable option blocks from the ReCollect page sections."""
    options = []

    for section in item.get("sections", []):
        title = section.get("title")
        included_from = section.get("included_from")
        row_text = []

        for row in section.get("rows", []):
            if row.get("type") == "html":
                html = row.get("html") or row.get("value") or ""
                cleaned = strip_html(html)
                if cleaned:
                    row_text.append(cleaned)

        if title or included_from or row_text:
            options.append({
                "section_title": title,
                "included_from": included_from,
                "text": " ".join(row_text),
            })

    return options


def is_boilerplate_option(text):
    return BOILERPLATE_RECYCLING_TEXT in (text or "").lower()


def infer_streams_for_option(option):
    template_text = " ".join([
        str(option.get("included_from") or ""),
        str(option.get("section_title") or ""),
    ]).lower()
    option_text = "" if is_boilerplate_option(option.get("text")) else option.get("text", "").lower()

    streams = []
    for stream, terms in TEMPLATE_STREAM_RULES.items():
        if any(term in template_text for term in terms):
            streams.append(stream)

    # Free text is only used for positive instructions. Do not infer garbage
    # or recycling from general text because DC often says things like
    # "do not dispose of batteries in the trash or recycling."
    for stream, terms in TEXT_STREAM_RULES.items():
        if any(term in option_text for term in terms):
            streams.append(stream)

    return list(dict.fromkeys(streams))


def primary_stream(streams):
    for stream in STREAM_PRIORITY:
        if stream in streams:
            return stream
    return "other"


def add_condition(conditions, stream, text):
    if text and text not in conditions[stream]:
        conditions[stream].append(text)


def normalize_item(item):
    title = get_variable(item, "title") or item.get("caption", "")
    synonyms_raw = get_variable(item, "synonyms")
    synonyms = [s.strip() for s in synonyms_raw.split(",") if s.strip()]

    dropoff_instructions = strip_html(get_variable(item, "dropoff_instructions"))
    special_instructions = strip_html(get_variable(item, "special_instructions"))
    options = extract_options(item)

    streams = []
    condition_lists = defaultdict(list)

    for option in options:
        option_streams = infer_streams_for_option(option)
        if is_boilerplate_option(option.get("text")):
            condition_text = ""
        else:
            condition_text = option.get("text", "")

        for stream in option_streams:
            streams.append(stream)
            add_condition(condition_lists, stream, condition_text)

    if dropoff_instructions:
        streams.append("dropoff")
        add_condition(condition_lists, "dropoff", dropoff_instructions)
        if "hazardous" in dropoff_instructions.lower():
            streams.append("hazardous")
            add_condition(condition_lists, "hazardous", dropoff_instructions)

    if special_instructions:
        streams.append("special")
        add_condition(condition_lists, "special", special_instructions)
        if "hazardous" in special_instructions.lower():
            streams.append("hazardous")
            add_condition(condition_lists, "hazardous", special_instructions)

    streams = [stream for stream in STREAM_PRIORITY if stream in set(streams)]
    primary = primary_stream(streams)
    non_boilerplate_options = [opt["text"] for opt in options if opt.get("text") and not is_boilerplate_option(opt.get("text"))]
    explanation = next((text for text in [special_instructions, dropoff_instructions, *non_boilerplate_options] if text), "")

    return {
        "item_id": str(item.get("id")),
        "item_name": title,
        "page_name": item.get("page_name"),
        "synonyms": synonyms,
        "disposal_streams": streams,
        "disposal_conditions": {stream: " ".join(parts) for stream, parts in condition_lists.items() if parts},
        "explanation": explanation,
        "category_path": ["Zero Waste DC What Goes Where", STREAM_DISPLAY.get(primary, "Uncategorized")],
        "source_reference": f"https://api.recollect.net/api/areas/WashingtonDC/services/220/pages/en-US/{item.get('id')}.json",
        "city_jurisdiction": "Washington, DC",
        "recollect_options": options,
    }


items = [normalize_item(item) for item in raw_items if item.get("page_type") == "material"]
no_stream = sum(1 for item in items if not item["disposal_streams"])
print(f"Transformed {len(items)} material records ({no_stream} with no inferred disposal stream)")
items[:2]

Transformed 468 material records (6 with no inferred disposal stream)


[{'item_id': '269984',
  'item_name': 'Acids',
  'page_name': 'acids',
  'synonyms': ['acid',
   'rust remover',
   'household hazardous waste',
   'hazardous'],
  'disposal_streams': ['hazardous', 'dropoff'],
  'disposal_conditions': {'hazardous': 'There is no curbside collection for this household hazardous waste.',
   'dropoff': 'There is no curbside collection for this household hazardous waste.'},
  'explanation': 'There is no curbside collection for this household hazardous waste.',
  'category_path': ['Zero Waste DC What Goes Where',
   'Household hazardous waste'],
  'source_reference': 'https://api.recollect.net/api/areas/WashingtonDC/services/220/pages/en-US/269984.json',
  'city_jurisdiction': 'Washington, DC',
  'recollect_options': [{'section_title': None,
    'included_from': None,
    'text': 'For a comprehensive list of what can be recycled in the District, please refer to this list .'},
   {'section_title': None,
    'included_from': None,
    'text': 'There is no curb

## 3. Flatten into a DataFrame

The records now have the same core shape as the Seattle notebook: item name, synonyms, disposal streams, per-stream instructions, explanation, source URL, and jurisdiction. A `primary_stream` column chooses one stream for charts when an item has several valid options.

In [5]:

items_df = pd.json_normalize(items)
items_df["primary_stream"] = items_df["disposal_streams"].apply(primary_stream)

print("Shape:", items_df.shape)
print("\nItems per primary disposal stream:")
print(items_df["primary_stream"].value_counts().to_string())
items_df[["item_name", "primary_stream", "disposal_streams", "category_path"]].head(10)

Shape: (468, 18)

Items per primary disposal stream:
primary_stream
garbage      149
hazardous    109
recycling     83
dropoff       58
compost       40
special       21
other          6
donate         2


,item_name,primary_stream,disposal_streams,category_path
0,Acids,hazardous,"[hazardous, dropoff]","[Zero Waste DC What Goes Where, Household haza..."
1,Aerosol can (empty),recycling,[recycling],"[Zero Waste DC What Goes Where, Recycling]"
2,Aerosol can (full or partially full),hazardous,"[hazardous, dropoff]","[Zero Waste DC What Goes Where, Household haza..."
3,Air conditioner,special,"[special, donate]","[Zero Waste DC What Goes Where, Special collec..."
4,air mattress,special,[special],"[Zero Waste DC What Goes Where, Special collec..."
5,Air pillows,garbage,"[garbage, dropoff, special]","[Zero Waste DC What Goes Where, Trash]"
6,Aluminum baking tray,recycling,[recycling],"[Zero Waste DC What Goes Where, Recycling]"
7,Aluminum cap,garbage,[garbage],"[Zero Waste DC What Goes Where, Trash]"
8,Aluminum foil,recycling,[recycling],"[Zero Waste DC What Goes Where, Recycling]"
9,Aluminum pie plate,recycling,[recycling],"[Zero Waste DC What Goes Where, Recycling]"


## 4. Visualize the disposal tree

DC's source data does not expose a material taxonomy like Seattle's category tree. To keep the same visual pattern, this sunburst uses the inferred primary disposal stream as the middle level and each item as a leaf.

In [6]:

labels = ["Zero Waste DC What Goes Where"]
parents = [""]
ids = ["root"]
hover = ["Washington, DC material guidance"]
colours = ["#dee2e6"]

for stream in STREAM_PRIORITY + ["other"]:
    matching = [item for item in items if primary_stream(item["disposal_streams"]) == stream]
    if not matching:
        continue
    stream_id = f"stream/{stream}"
    labels.append(STREAM_DISPLAY[stream])
    parents.append("root")
    ids.append(stream_id)
    hover.append(f"{STREAM_DISPLAY[stream]}: {len(matching)} items")
    colours.append(STREAM_COLOUR[stream])

    for item in matching:
        item_id = f"item/{item['item_id']}"
        labels.append(item["item_name"])
        parents.append(stream_id)
        ids.append(item_id)
        stream_text = ", ".join(item["disposal_streams"]) or "none inferred"
        hover.append(f"{item['item_name']}<br>Streams: {stream_text}")
        colours.append(STREAM_COLOUR[stream])

fig = go.Figure(go.Sunburst(
    labels=labels,
    parents=parents,
    ids=ids,
    hovertext=hover,
    hoverinfo="text",
    marker={"colors": colours},
    maxdepth=3,
))
fig.update_layout(
    title="Washington, DC What Goes Where items by inferred disposal stream",
    width=900,
    height=700,
    margin=dict(t=60, l=10, r=10, b=10),
)
fig.show()

## 5. Distribution across disposal streams

This bar chart uses each item's primary stream, matching the Seattle notebook's approach for multi-stream items.

In [7]:

counts = items_df["primary_stream"].value_counts().reindex(STREAM_PRIORITY + ["other"]).dropna().reset_index()
counts.columns = ["stream", "count"]

bar = go.Figure(go.Bar(
    x=[STREAM_DISPLAY[s] for s in counts["stream"]],
    y=counts["count"],
    marker_color=[STREAM_COLOUR[s] for s in counts["stream"]],
    text=counts["count"],
    textposition="outside",
))
bar.update_layout(
    title="Washington, DC items per primary disposal stream",
    xaxis_title="Primary disposal stream",
    yaxis_title="Number of items",
    width=900,
    height=450,
    plot_bgcolor="#fafafa",
)
bar.show()

## 6. Item lookup - the portable use case

This is the same core use case as the Seattle notebook: look up an item by name or synonym and return disposal streams, instructions, and a source reference.

In [8]:

def lookup(query, items=items, limit=5):
    """Find matching items by item name or synonym."""
    q = query.lower().strip()
    hits = []

    for item in items:
        haystack = [item["item_name"].lower()] + [syn.lower() for syn in item.get("synonyms", [])]
        if any(q == value or q in value for value in haystack):
            hits.append(item)

    return hits[:limit]


def explain(query):
    results = lookup(query)
    if not results:
        print(f"No match for '{query}'.")
        return

    for item in results:
        print(f"\n-> {item['item_name'].upper()}")
        print(f"   Category:  {' > '.join(item['category_path'])}")
        print(f"   Streams:   {', '.join(item['disposal_streams']) or '(none inferred)'}")
        if item["explanation"]:
            print(f"   Summary:   {item['explanation'][:300]}")
        for stream, text in item["disposal_conditions"].items():
            print(f"   - {stream.title()}: {text[:250]}")
        print(f"   Source:    {item['source_reference']}")


explain("battery")
explain("pizza box")
explain("plastic bag")


-> BATTERY (CAR)
   Category:  Zero Waste DC What Goes Where > Household hazardous waste
   Streams:   hazardous, dropoff
   Summary:   There is no curbside collection for this household hazardous waste.
   - Hazardous: There is no curbside collection for this household hazardous waste.
   - Dropoff: There is no curbside collection for this household hazardous waste.
   Source:    https://api.recollect.net/api/areas/WashingtonDC/services/220/pages/en-US/270002.json

-> BATTERY (NON-RECHARGEABLE)
   Category:  Zero Waste DC What Goes Where > Household hazardous waste
   Streams:   hazardous, dropoff
   Summary:   All batteries must now be disposed of carefully in DC and all batteries may not be disposed of in the trash. Please read about DC's Battery Disposal Ban here: https://code.dccouncil.gov/us/dc/council/code/sections/8-771.09 If you need to dispose of a battery, all types of batteries, please locate a 
   - Hazardous: DPW's Special Waste Collection Events - Household Hazardous Wa

## 7. Save a portable snapshot

The raw ReCollect snapshot is saved separately from the transformed portable data. The portable output is `data/dc_items_portable.json`, leaving the older `data/dc_items.json` generated by `dc_recollect_pull.py` untouched.

In [9]:

snapshot = {
    "metadata": {
        "source": "ReCollect API for Zero Waste DC What Goes Where",
        "source_url": source_url,
        "source_mode": source_mode,
        "city_jurisdiction": "Washington, DC",
        "schema_version": "1.0-dc-portable",
        "item_count": len(items),
        "generated_at": datetime.now(timezone.utc).isoformat(),
    },
    "items": items,
}

DATA_DIR.mkdir(exist_ok=True)
with PORTABLE_SNAPSHOT.open("w", encoding="utf-8") as f:
    json.dump(snapshot, f, indent=2, ensure_ascii=False)

size_kb = PORTABLE_SNAPSHOT.stat().st_size / 1024
print(f"Saved {PORTABLE_SNAPSHOT} - {len(items)} items, {size_kb:.1f} KB")

Saved dc/data/dc_items_portable.json - 468 items, 1242.1 KB


## 8. Validation checks

These lightweight checks confirm that the DC notebook creates a usable data artifact with the same core fields as the Seattle notebook.

In [10]:

REQUIRED_FIELDS = {
    "item_id", "item_name", "synonyms", "disposal_streams", "disposal_conditions",
    "explanation", "category_path", "source_reference", "city_jurisdiction",
}
VALID_STREAMS = set(STREAM_PRIORITY)

missing_fields = [
    item["item_id"]
    for item in items
    if REQUIRED_FIELDS - set(item.keys())
]
ids = [item["item_id"] for item in items]
unknown_streams = sorted({
    stream
    for item in items
    for stream in item["disposal_streams"]
    if stream not in VALID_STREAMS
})
lookup_checks = {
    "battery": len(lookup("battery")),
    "pizza box": len(lookup("pizza box")),
    "plastic bag": len(lookup("plastic bag")),
}

assert len(items) >= 400, f"Expected at least 400 DC material records, got {len(items)}"
assert not missing_fields, f"Items missing required fields: {missing_fields[:10]}"
assert len(ids) == len(set(ids)), "Duplicate item IDs found"
assert not unknown_streams, f"Unknown streams found: {unknown_streams}"
assert all(count > 0 for count in lookup_checks.values()), lookup_checks
assert PORTABLE_SNAPSHOT.exists(), f"Missing output file: {PORTABLE_SNAPSHOT}"

print("Validation passed")
print(f"Items: {len(items)}")
print(f"Items with no inferred stream: {sum(1 for item in items if not item['disposal_streams'])}")
print("Lookup checks:", lookup_checks)

Validation passed
Items: 468
Items with no inferred stream: 6
Lookup checks: {'battery': 5, 'pizza box': 2, 'plastic bag': 5}


## Summary

- The DC notebook follows the same workflow as the Seattle notebook: fetch, normalize, inspect in a DataFrame, visualize, look up items, save JSON, and validate.
- DC's ReCollect data requires stream inference because the source does not expose Seattle-style stream fields directly.
- The output `data/dc_items_portable.json` is the DC portable artifact and can power a lookup UI or API the same way Seattle's `data/items.json` does.